In [13]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
SPLIT_DIR = PROJECT_ROOT / "data" / "interim" / "splits"

train_df = pd.read_csv(
    SPLIT_DIR / "train.csv",
    parse_dates=["order date (DateOrders)"]
)

val_df = pd.read_csv(
    SPLIT_DIR / "validation.csv",
    parse_dates=["order date (DateOrders)"]
)

test_df = pd.read_csv(
    SPLIT_DIR / "test.csv",
    parse_dates=["order date (DateOrders)"]
)

In [14]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

print(train_df["order date (DateOrders)"].dtype)

(46026, 15)
(9863, 15)
(9863, 15)
datetime64[us]


In [15]:
for split_df in [train_df, val_df, test_df]:
    split_df["order_hour"] = split_df["order date (DateOrders)"].dt.hour
    split_df["order_dayofweek"] = split_df["order date (DateOrders)"].dt.dayofweek
    split_df["order_month"] = split_df["order date (DateOrders)"].dt.month

In [20]:
train_df[
    [
        "order date (DateOrders)",
        "order_hour",
        "order_dayofweek",
        "order_month",
    ]
].head()

,order date (DateOrders),order_hour,order_dayofweek,order_month
0,2015-01-01 00:00:00,0,3,1
1,2015-01-01 00:21:00,0,3,1
2,2015-01-01 01:03:00,1,3,1
3,2015-01-01 01:24:00,1,3,1
4,2015-01-01 02:06:00,2,3,1


In [21]:
RARE_THRESHOLD = 50

country_counts = train_df["Order Country"].value_counts()

frequent_countries = country_counts[
    country_counts >= RARE_THRESHOLD
].index

for split_df in [train_df, val_df, test_df]:
    split_df["Order Country"] = (
        split_df["Order Country"]
        .where(
            split_df["Order Country"].isin(frequent_countries),
            "Other"
        )
    )

In [22]:
print("Train countries:", train_df["Order Country"].nunique())
print("Validation countries:", val_df["Order Country"].nunique())
print("Test countries:", test_df["Order Country"].nunique())

print("\nOther counts:")
print("Train:", (train_df["Order Country"] == "Other").sum())
print("Validation:", (val_df["Order Country"] == "Other").sum())
print("Test:", (test_df["Order Country"] == "Other").sum())

Train countries: 82
Validation countries: 33
Test countries: 30

Other counts:
Train: 1233
Validation: 90
Test: 114


In [23]:
CATEGORICAL_FEATURES = [
    "Type",
    "Customer Segment",
    "Customer State",
    "Order Country",
    "Order Region",
    "Shipping Mode",
]

NUMERIC_FEATURES = [
    "total_quantity",
    "total_discount",
    "num_unique_products",
    "num_unique_categories",
    "num_unique_departments",
    "order_hour",
    "order_dayofweek",
    "order_month",
]

TARGET = "Late_delivery_risk"

TECHNICAL_COLUMNS = [
    "Order Id",
    "Customer Id",
    "order date (DateOrders)",
]

In [24]:
expected_features = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
    + TECHNICAL_COLUMNS
    + [TARGET]
)

missing_columns = [
    col for col in expected_features
    if col not in train_df.columns
]

extra_columns = [
    col for col in train_df.columns
    if col not in expected_features
]

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)
print("Total train columns:", len(train_df.columns))

Missing columns: []
Extra columns: []
Total train columns: 18


In [25]:
X_train = train_df[CATEGORICAL_FEATURES + NUMERIC_FEATURES].copy()
X_val = val_df[CATEGORICAL_FEATURES + NUMERIC_FEATURES].copy()
X_test = test_df[CATEGORICAL_FEATURES + NUMERIC_FEATURES].copy()

y_train = train_df[TARGET].copy()
y_val = val_df[TARGET].copy()
y_test = test_df[TARGET].copy()

In [26]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (46026, 14)
X_val: (9863, 14)
X_test: (9863, 14)
y_train: (46026,)
y_val: (9863,)
y_test: (9863,)


In [28]:
%pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 8.8 MB/s  0:00:00 eta 0:00:01m
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 7.9 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
    ]
)

In [30]:
X_train_prepared = preprocessor.fit_transform(X_train)

X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)

In [31]:
print("X_train_prepared:", X_train_prepared.shape)
print("X_val_prepared:", X_val_prepared.shape)
print("X_test_prepared:", X_test_prepared.shape)

X_train_prepared: (46026, 168)
X_val_prepared: (9863, 168)
X_test_prepared: (9863, 168)


In [32]:
feature_names = preprocessor.get_feature_names_out()

print("Number of prepared features:", len(feature_names))
print(feature_names)

Number of prepared features: 168
['categorical__Type_CASH' 'categorical__Type_DEBIT'
 'categorical__Type_PAYMENT' 'categorical__Type_TRANSFER'
 'categorical__Customer Segment_Consumer'
 'categorical__Customer Segment_Corporate'
 'categorical__Customer Segment_Home Office'
 'categorical__Customer State_AL' 'categorical__Customer State_AR'
 'categorical__Customer State_AZ' 'categorical__Customer State_CA'
 'categorical__Customer State_CO' 'categorical__Customer State_CT'
 'categorical__Customer State_DC' 'categorical__Customer State_DE'
 'categorical__Customer State_FL' 'categorical__Customer State_GA'
 'categorical__Customer State_HI' 'categorical__Customer State_IA'
 'categorical__Customer State_ID' 'categorical__Customer State_IL'
 'categorical__Customer State_IN' 'categorical__Customer State_KS'
 'categorical__Customer State_KY' 'categorical__Customer State_LA'
 'categorical__Customer State_MA' 'categorical__Customer State_MD'
 'categorical__Customer State_MI' 'categorical__Customer 

In [34]:
from scipy import sparse

print(type(X_train_prepared))
print("Is sparse:", sparse.issparse(X_train_prepared))
print("dtype:", X_train_prepared.dtype)

<class 'scipy.sparse._csr.csr_matrix'>
Is sparse: True
dtype: float64


In [ ]:
import numpy as np

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [35]:

print("NaN in X_train_prepared:", np.isnan(X_train_prepared.data).sum())
print("NaN in X_val_prepared:", np.isnan(X_val_prepared.data).sum())
print("NaN in X_test_prepared:", np.isnan(X_test_prepared.data).sum())

print("Inf in X_train_prepared:", np.isinf(X_train_prepared.data).sum())
print("Inf in X_val_prepared:", np.isinf(X_val_prepared.data).sum())
print("Inf in X_test_prepared:", np.isinf(X_test_prepared.data).sum())

print("\nTarget values:")
print("Train:", sorted(y_train.unique()))
print("Validation:", sorted(y_val.unique()))
print("Test:", sorted(y_test.unique()))

NaN in X_train_prepared: 0
NaN in X_val_prepared: 0
NaN in X_test_prepared: 0
Inf in X_train_prepared: 0
Inf in X_val_prepared: 0
Inf in X_test_prepared: 0

Target values:
Train: [np.int64(0), np.int64(1)]
Validation: [np.int64(0), np.int64(1)]
Test: [np.int64(0), np.int64(1)]


## Baseline model

### 1- LogisticRegression 

In [36]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train_prepared, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [37]:
y_val_pred = logreg.predict(X_val_prepared)
y_val_proba = logreg.predict_proba(X_val_prepared)[:, 1]

In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_val_proba))

Accuracy: 0.7008009733346852
Precision: 0.8362930077691454
Recall: 0.5607441860465117
F1: 0.6713442476890522
ROC-AUC: 0.7429475604195166


In [39]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_val, y_val_pred)
print(cm)

print()
print(classification_report(y_val, y_val_pred, digits=3))

[[3898  590]
 [2361 3014]]

              precision    recall  f1-score   support

           0      0.623     0.869     0.725      4488
           1      0.836     0.561     0.671      5375

    accuracy                          0.701      9863
   macro avg      0.730     0.715     0.698      9863
weighted avg      0.739     0.701     0.696      9863

